In [1]:
from cesnet_datazoo.datasets import CESNET_QUIC22
from cesnet_datazoo.config import DatasetConfig, AppSelection, ValidationApproach

dataset = CESNET_QUIC22("~/datasets/CESNET-QUIC22/", size="XS")

common_params = {
    "dataset" : dataset,
    "apps_selection" : AppSelection.ALL_KNOWN,
    "test_period_name" : "W-2022-44",
    "val_approach": ValidationApproach.SPLIT_FROM_TRAIN,
    "train_val_split_fraction": 0.2
}

dataset_config = DatasetConfig(**common_params)
dataset.set_dataset_config_and_initialize(dataset_config)
train_dataframe = dataset.get_train_df(flatten_ppi=True)
val_dataframe = dataset.get_val_df(flatten_ppi=True)
test_dataframe = dataset.get_test_df(flatten_ppi=True)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/cesnet_datazoo/config.py:341: UserWarning: Some test dates (20221031) are before or equal to the last train date (20221106). This might lead to improper evaluation and should be avoided.
  warnings.warn(f"Some test dates ({min(test_dates).strftime('%Y%m%d')}) are before or equal to the last train date ({max(train_dates).strftime('%Y%m%d')}). This might lead to improper evaluation and should be avoided.")


Loading data from dataloader


100%|██████████| 8162/8162 [00:13<00:00, 621.92it/s] 


Loading data from dataloader


100%|██████████| 192/192 [00:06<00:00, 30.93it/s]


Loading data from dataloader


100%|██████████| 957/957 [00:10<00:00, 92.24it/s] 


In [2]:
import numpy as np

def create_balanced_test_data(nfeatures, test_dataframe, nfrom_class = 100):
    grouped = test_dataframe.groupby("APP")

    X_arr = np.ndarray(shape = (nfrom_class * len(grouped), nfeatures))
    y_arr = np.ndarray(shape = (nfrom_class * len(grouped),))

    for index, i in enumerate(grouped):
        X_temp = i[1].drop(columns="APP").to_numpy()
        y_temp = i[1]["APP"].to_numpy()

        X_arr[index*nfrom_class:(index * nfrom_class) + nfrom_class] = X_temp[:nfrom_class]
        y_arr[index*nfrom_class:(index * nfrom_class) + nfrom_class] = y_temp[:nfrom_class]

    return (X_arr, y_arr)

In [21]:
import torch
import torch.nn as nn
import torch.nn.functional as F 
import torch.optim as optim

import random as rd
from tqdm import tqdm
from collections import deque

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
depth = 15

N_STATE = train_dataframe.shape[1] - 1
N_OUT = len(train_dataframe.groupby('APP'))

(X_balanced, y_balanced) = create_balanced_test_data(N_STATE, test_dataframe, 100)
X_chosen = np.ndarray(shape=(1000, N_STATE))
y_chosen = np.ndarray(shape=(1000))
X = train_dataframe.drop(columns="APP").to_numpy()
y = train_dataframe["APP"].to_numpy()

X_big_test = test_dataframe.drop(columns="APP").to_numpy()[:10000]
y_big_test = test_dataframe["APP"].to_numpy()[:10000]
chosen_i = 0

from sys import platform
device = 'cuda' if platform == 'win32' else 'mps'

def big_test(steps, total):
    clf = RandomForestClassifier(max_depth=depth, n_jobs=-1)
    clf.fit(X_chosen[:steps], y_chosen[:steps])

    predict_arr = clf.predict(X_big_test)

    print(f"q_learning_f1: {f1_score(y_big_test, predict_arr, average='macro'):.4f}" + "\n")

    val = 0
    for _ in range(3):
        clf = RandomForestClassifier(max_depth=depth, n_jobs=-1)
        indices = np.random.choice(total, steps, replace=False)

        clf.fit(X[indices], y[indices])
        
        predict_arr = clf.predict(X_big_test)

        val += f1_score(y_big_test, predict_arr, average='macro')

    val /= 3
    print(f"random_learning_f1: {val:.4f}" + "\n")

    clf = RandomForestClassifier(max_depth = depth, n_jobs=-1)
    clf.fit(X[:total], y[:total])
    
    predict_arr = clf.predict(X_big_test)
    
    print(f"total_learning_f1: {f1_score(y_big_test, predict_arr, average='macro'):.4f}" + "\n")


class Memory():
   def __init__(self):
      self.buffer = deque(maxlen = 10000)

   def push(self, last_state, action, reward, next_state):
      self.buffer.append((last_state, action, reward, next_state))

   def sample(self, batch_size):
      last_state, action, reward, next_state = zip(*rd.sample(self.buffer, batch_size))
      return last_state, action, reward, next_state
   
   def __len__(self):
      return len(self.buffer)

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = nn.Linear(N_STATE, 128)
        self.l2 = nn.Linear(128, 32)
        self.l3 = nn.Linear(32, 2)

    def forward(self, x):
        x = F.relu(self.l1(x))
        x = F.relu(self.l2(x))
        x = self.l3(x)

        return x
        
    def act(self, state, epsilon):
        if rd.random() > epsilon:
            state = torch.Tensor(np.float32(state)).to(device)
            q_value = self.forward(state)
            action = q_value.argmax().item()

        else:
            action = rd.randint(0, 1)

        return action
    
net = Net().to(device)
optimizer = optim.AdamW(net.parameters(), lr = 1e-3)
last_f1 = 0
memory = Memory()

def compute_reward(n_chosen):
    clf = RandomForestClassifier(max_depth = depth, n_jobs=-1)
    clf.fit(X_chosen[:n_chosen], y_chosen[:n_chosen])
    predict_arr = clf.predict(X_balanced)

    return f1_score(y_balanced, predict_arr, average="macro") - last_f1

def compute_loss(memory, batch_size):
    last_state, action, reward, next_state = memory.sample(batch_size)

    last_state = torch.Tensor(np.float32(last_state)).to(device)
    next_state = torch.Tensor(np.float32(next_state)).to(device)
    action = torch.LongTensor(action).to(device)
    reward = torch.Tensor(reward).to(device)

    q_old = net(last_state)
    q_new = net(next_state)

    q_old = q_old.gather(1, action.unsqueeze(1)).squeeze(1)
    q_new = q_new.max(1)[0]

    q_expected = reward + 0.99 * q_new

    loss = (q_old - q_expected.data).pow(2).mean()

    return loss

X_chosen[chosen_i] = X[0]
y_chosen[chosen_i] = y[0]
chosen_i += 1

for step in tqdm(range(1, 2000)):

    action = net.act(X[step], 0.1)

    if action:
        X_chosen[chosen_i] = X[step]
        y_chosen[chosen_i] = y[step]
        chosen_i += 1

    reward = compute_reward(chosen_i)
    last_f1 = reward
    memory.push(X[step-1], action, reward, X[step])

    if len(memory) > 32:
        loss = compute_loss(memory, 32)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

big_test(chosen_i, 1999)



 17%|█▋        | 698/3999 [02:21<11:09,  4.93it/s]


KeyboardInterrupt: 

In [19]:
print(chosen_i)

555
